In [4]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

base_path = '/content/drive/MyDrive/Proyecto_arroz/Conjunto_datos/'
archivos = ['arroz_1(me).csv', 'arroz_2.csv', 'arroz_3.csv', 'arroz_4.csv', 'arroz_5.csv', 'arroz_6.csv']

lista_dfs = []

print("--- Cargando y Unificando Archivos ---")
for archivo in archivos:
    ruta = os.path.join(base_path, archivo)
    try:
        df_temp = pd.read_csv(ruta, sep=None, engine='python')
        if len(df_temp.columns) <= 1:
            df_temp = df_temp.iloc[:,0].str.split(';', expand=True)

        df_temp.columns = [f'p{i}' for i in range(len(df_temp.columns)-1)] + ['target']
        lista_dfs.append(df_temp)
        print(f"{archivo} cargado correctamente.")
    except Exception as e:
        print(f"Error en {archivo}: {e}")

if lista_dfs:
    df_final = pd.concat(lista_dfs, ignore_index=True)

    X = df_final.drop('target', axis=1).apply(pd.to_numeric, errors='coerce').fillna(0)
    y = df_final['target'].astype(str).str.strip() # Aseguramos que las etiquetas sean texto limpio

    counts = y.value_counts()
    clases_validas = counts[counts > 1].index
    mask = y.isin(clases_validas)
    X, y = X[mask], y[mask]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

    print(f"\n Entrenando con {len(X_train)} muestras. Comparando resultados...\n")

    modelos = {
        "SVM (Lineal)": SVC(kernel='linear'),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "KNN (k=3)": KNeighborsClassifier(n_neighbors=3),
        "Naive Bayes": GaussianNB()
    }

    resultados = {}

    for nombre, modelo in modelos.items():
        modelo.fit(X_train, y_train)
        predicciones = modelo.predict(X_test)
        acc = accuracy_score(y_test, predicciones)
        resultados[nombre] = acc
        print(f"{nombre:<15} | Accuracy: {acc:.4f}")

    mejor_modelo = max(resultados, key=resultados.get)
    print(f"\n El algoritmo mas capacitado es: {mejor_modelo} con {resultados[mejor_modelo]:.4f} de precisión.")

else:
    print("No se cargó ningún dato. Revisa las rutas en tu Drive.")

--- Cargando y Unificando Archivos ---
arroz_1(me).csv cargado correctamente.
arroz_2.csv cargado correctamente.
arroz_3.csv cargado correctamente.
arroz_4.csv cargado correctamente.
arroz_5.csv cargado correctamente.
arroz_6.csv cargado correctamente.

 Entrenando con 144 muestras. Comparando resultados...

SVM (Lineal)    | Accuracy: 0.5556
Random Forest   | Accuracy: 0.6667
KNN (k=3)       | Accuracy: 0.7222
Naive Bayes     | Accuracy: 0.6944

 El algoritmo mas capacitado es: KNN (k=3) con 0.7222 de precisión.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
